# Pipeline Satu Lapis — Baseline Arsitektur

Notebook ini adalah **baseline** untuk perbandingan arsitektur.
Semua proses dari data mentah sampai clustering dilakukan dalam satu alur,
tanpa pemisahan Bronze/Silver/Gold layer.

**Tujuan:** Menunjukkan bahwa Medallion Architecture lebih terstruktur
dan efisien dibanding pipeline tanpa pemisahan layer.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
import time, os

BRONZE_PATH = '/content/drive/MyDrive/ABD/data/bronze/ghgrp_bronze.parquet'
start_time  = time.time()

df = pd.read_parquet(BRONZE_PATH)
print(f'Data dibaca: {len(df):,} baris')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data dibaca: 66,881 baris


In [ ]:
# Cleaning — semua dalam satu sel tanpa checkpoint
df = df.dropna(subset=['Total reported direct emissions'])
df = df[df['Total reported direct emissions'] > 0]
df = df.drop_duplicates(subset=['Facility Id', 'year'])
df['sector_clean'] = df['Industry Type (sectors)'].str.split(',').str[0].str.strip()
df['log_emissions'] = np.log(df['Total reported direct emissions'] + 1)

for gas in ['CO2 emissions (non-biogenic) ', 'Methane (CH4) emissions ', 'Nitrous Oxide (N2O) emissions ']:
    if gas in df.columns:
        df[gas] = df[gas].fillna(df[gas].median())

print(f'Setelah cleaning: {len(df):,} baris')

Setelah cleaning: 66,568 baris


In [ ]:
# Feature engineering + clustering — langsung tanpa layer Gold
facility = df.groupby('Facility Id').agg(
    avg_emissions=('Total reported direct emissions', 'mean'),
    avg_log_emissions=('log_emissions', 'mean'),
    years_reported=('year', 'count'),
    sector_clean=('sector_clean', 'first')
).reset_index()

X = facility[['avg_log_emissions', 'years_reported']].values
X_scaled = StandardScaler().fit_transform(X)

km = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = km.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)
t_total = time.time() - start_time

print(f'Waktu total       : {t_total:.3f} detik')
print(f'Silhouette Score  : {sil:.4f}')
print(f'Davies-Bouldin    : {dbi:.4f}')
print()
print('Catatan: Pipeline ini tidak punya checkpoint.')
print('Kalau ada error di cleaning, harus mulai ulang dari awal.')
print('Tidak ada cara untuk reprocess hanya bagian Gold tanpa rerun semua.')

Waktu total       : 16.513 detik
Silhouette Score  : 0.5505
Davies-Bouldin    : 0.6135

Catatan: Pipeline ini tidak punya checkpoint.
Kalau ada error di cleaning, harus mulai ulang dari awal.
Tidak ada cara untuk reprocess hanya bagian Gold tanpa rerun semua.
